## Portfolio Optimization Using Modern Portfolio Theory

### My Real Positions — Fidelity Brokerage Accounts

This notebook applies **Modern Portfolio Theory (MPT)** to the real equity positions stored
in the MySQL `Portfolio_Positions` and `equity_historical` tables.

**Key differences from the example crypto notebook:**

| Aspect | Crypto Example | This Notebook |
|--------|---------------|---------------|
| Data source | Live API (yfinance) | MySQL cache (no API calls) |
| Assets | 10 hard-coded crypto tickers | 80+ real equity holdings |
| Frequency | 365 days/year (crypto) | 252 trading days/year (equities) |
| Accounts | Single portfolio | Multiple accounts (taxable, retirement, 529, HSA) |
| Baseline | No existing allocation | Compare current vs. optimal |

**Strategy:**
1. Load current positions and historical prices from MySQL
2. Build a daily close price matrix (last 3 years for robust statistics)
3. Calculate expected returns (mean historical) and covariance (Ledoit-Wolf shrinkage)
4. Find optimal portfolios: **Max Sharpe**, **Min Volatility**, **Efficient Risk**
5. Compare current allocation against optimal — identify rebalancing opportunities
6. Visualize the efficient frontier with current portfolio plotted on it

> **Author:** Prashant Rajoria | **Date:** 2026-02-21 | **Branch:** `openbb_learning`

### 1. Setup & Imports

In [ ]:
import sys
sys.stdout.reconfigure(encoding="utf-8", errors="replace")

import warnings
warnings.filterwarnings("ignore", category=FutureWarning)

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import pymysql

from pypfopt import expected_returns, EfficientFrontier, CLA
from pypfopt import CovarianceShrinkage
from pypfopt import plotting as pplt
from pypfopt.discrete_allocation import DiscreteAllocation, get_latest_prices

# Display settings
pd.set_option("display.max_rows", 100)
pd.set_option("display.float_format", "{:.4f}".format)

print("All imports successful ✓")

### 2. Connect to MySQL and Load Current Positions

We read directly from the `Portfolio_Positions` table in `openbb_fmp_cache_test`.
Each row is a cost-basis lot; we aggregate by symbol to get total holdings.

In [ ]:
# --- Database connection ---
DB_CONFIG = dict(
    host="localhost",
    port=3306,
    user="fmp_user",
    password="fmp_password",
    database="openbb_fmp_cache_test",
    cursorclass=pymysql.cursors.DictCursor,
    charset="utf8mb4",
)

conn = pymysql.connect(**DB_CONFIG)

# --- Load positions (aggregate lots by symbol per account) ---
positions_sql = """
SELECT
    account_name,
    symbol,
    description,
    SUM(quantity)          AS total_qty,
    SUM(current_value)     AS total_value,
    SUM(cost_basis_total)  AS total_cost,
    SUM(total_gain_loss)   AS total_gl,
    snapshot_date
FROM Portfolio_Positions
WHERE snapshot_date = (SELECT MAX(snapshot_date) FROM Portfolio_Positions)
  AND symbol NOT IN ('Cash', 'Pending Activity')
  AND quantity > 0
GROUP BY account_name, symbol, description, snapshot_date
ORDER BY total_value DESC
"""

positions_df = pd.read_sql(positions_sql, conn)
print(f"Loaded {len(positions_df)} position-account rows")
print(f"Unique symbols: {positions_df['symbol'].nunique()}")
print(f"Accounts: {positions_df['account_name'].nunique()}")
print(f"Snapshot: {positions_df['snapshot_date'].iloc[0]}")
print(f"Total portfolio value: ${positions_df['total_value'].sum():,.2f}")

### 3. Build Portfolio-Level Allocation

Aggregate across all accounts to see the **total current allocation** per symbol.

In [ ]:
# Aggregate across all accounts for portfolio-wide view
portfolio = (
    positions_df
    .groupby("symbol")
    .agg(
        description=("description", "first"),
        total_qty=("total_qty", "sum"),
        total_value=("total_value", "sum"),
        total_cost=("total_cost", "sum"),
        total_gl=("total_gl", "sum"),
    )
    .sort_values("total_value", ascending=False)
)

portfolio["weight_pct"] = portfolio["total_value"] / portfolio["total_value"].sum() * 100
portfolio["gl_pct"] = np.where(
    portfolio["total_cost"] != 0,
    portfolio["total_gl"] / portfolio["total_cost"] * 100,
    0,
)

print(f"\n{'Symbol':<8} {'Description':<30} {'Value':>12} {'Weight%':>8} {'G/L%':>8}")
print("-" * 70)
for sym, row in portfolio.head(20).iterrows():
    print(f"{sym:<8} {row['description'][:30]:<30} ${row['total_value']:>10,.2f} "
          f"{row['weight_pct']:>7.2f}% {row['gl_pct']:>7.1f}%")

print(f"\n... showing top 20 of {len(portfolio)} positions")
print(f"Top 10 concentration: {portfolio['weight_pct'].head(10).sum():.1f}%")

### 4. Visualize Current Allocation

Treemap showing position sizes with gain/loss coloring.

In [ ]:
# Treemap of current allocation
treemap_df = portfolio.reset_index().copy()
treemap_df["label"] = treemap_df["symbol"] + "<br>" + treemap_df["weight_pct"].round(1).astype(str) + "%"

fig = px.treemap(
    treemap_df.head(30),
    path=["label"],
    values="total_value",
    color="gl_pct",
    color_continuous_scale="RdYlGn",
    color_continuous_midpoint=0,
    title="Current Portfolio Allocation (Top 30 Positions)",
)
fig.update_layout(width=900, height=600)
fig.show()

### 5. Load Historical Prices from Cache

Pull daily close prices from `equity_historical` for the last 3 years.
We need overlapping date ranges for all symbols to compute a valid covariance matrix.

In [ ]:
# Get the symbols that have historical data
portfolio_symbols = list(portfolio.index)

placeholders = ", ".join(["%s"] * len(portfolio_symbols))
history_sql = f"""
SELECT symbol, date, close
FROM equity_historical
WHERE symbol IN ({placeholders})
  AND date >= DATE_SUB(CURDATE(), INTERVAL 3 YEAR)
  AND close IS NOT NULL
  AND close > 0
ORDER BY symbol, date
"""

history_df = pd.read_sql(history_sql, conn, params=portfolio_symbols)
print(f"Loaded {len(history_df):,} price rows")
print(f"Symbols with history: {history_df['symbol'].nunique()} / {len(portfolio_symbols)}")
print(f"Date range: {history_df['date'].min()} to {history_df['date'].max()}")

# Check which symbols are missing
symbols_with_history = set(history_df["symbol"].unique())
missing = [s for s in portfolio_symbols if s not in symbols_with_history]
if missing:
    print(f"\n⚠ Symbols WITHOUT history (excluded from optimization): {missing}")

In [ ]:
# Pivot to wide format: date × symbol close prices
prices = (
    history_df
    .pivot(index="date", columns="symbol", values="close")
    .sort_index()
)

# Drop symbols with < 60% data coverage (too many missing days)
min_coverage = 0.6
coverage = prices.notna().mean()
valid_symbols = coverage[coverage >= min_coverage].index.tolist()
dropped = coverage[coverage < min_coverage].index.tolist()

if dropped:
    print(f"Dropped {len(dropped)} symbols with <{min_coverage*100:.0f}% coverage: {dropped}")

prices = prices[valid_symbols].dropna(how="all")

# Forward-fill small gaps (weekends already excluded, but holidays may leave NaN)
prices = prices.ffill(limit=5)

# Drop any remaining rows/cols with NaN
prices = prices.dropna(axis=1)

print(f"\nClean price matrix: {prices.shape[0]} days × {prices.shape[1]} symbols")
print(f"Date range: {prices.index.min()} to {prices.index.max()}")

# Show symbols available for optimization
opt_symbols = list(prices.columns)
print(f"Symbols for optimization: {len(opt_symbols)}")

### 6. Calculate Expected Returns & Covariance Matrix

Using **mean historical returns** (annualized, 252 trading days/year) and
**Ledoit-Wolf shrinkage** for a robust covariance estimate.

$$\mu_i = \frac{\sum r_i}{N} \times 252$$

$$\hat{\Sigma} = \alpha \cdot S + (1 - \alpha) \cdot F$$

where $S$ is the sample covariance, $F$ is the structured target, and $\alpha$ is
the shrinkage intensity (estimated by Ledoit-Wolf).

In [ ]:
# Expected returns — mean historical, annualized for equities (252 trading days)
mu = expected_returns.mean_historical_return(prices, frequency=252, compounding=True)

# Covariance matrix — Ledoit-Wolf shrinkage for stability
cov_matrix = CovarianceShrinkage(prices, frequency=252).ledoit_wolf()

print("Expected Annual Returns (top 10 / bottom 5):")
mu_sorted = mu.sort_values(ascending=False)
for sym in mu_sorted.head(10).index:
    print(f"  {sym:<8} {mu_sorted[sym]:>8.1%}")
print("  ...")
for sym in mu_sorted.tail(5).index:
    print(f"  {sym:<8} {mu_sorted[sym]:>8.1%}")

# Annual volatility per asset
vols = np.sqrt(np.diag(cov_matrix)) 
print(f"\nAverage annual volatility: {np.mean(vols):.1%}")
print(f"Volatility range: {np.min(vols):.1%} — {np.max(vols):.1%}")

### 7. Current Portfolio Performance Metrics

Calculate the expected return and volatility for the **current** allocation
so we can compare it against the optimized portfolios.

In [ ]:
# Build current weights vector (only for symbols in the optimization universe)
current_alloc = portfolio.loc[portfolio.index.isin(opt_symbols), "total_value"]
current_weights = current_alloc / current_alloc.sum()

# Reindex to match mu/cov ordering
current_weights = current_weights.reindex(mu.index).fillna(0)

# Current portfolio stats
current_return = current_weights @ mu
current_vol = np.sqrt(current_weights @ cov_matrix @ current_weights)
risk_free_rate = 0.045  # ~4.5% as of 2025-2026

current_sharpe = (current_return - risk_free_rate) / current_vol

print("=" * 50)
print("  CURRENT PORTFOLIO METRICS")
print("=" * 50)
print(f"  Expected Annual Return:  {current_return:>8.2%}")
print(f"  Annual Volatility:       {current_vol:>8.2%}")
print(f"  Sharpe Ratio:            {current_sharpe:>8.2f}")
print(f"  Risk-Free Rate:          {risk_free_rate:>8.2%}")
print("=" * 50)

### 8. Optimize: Maximum Sharpe Ratio Portfolio

The **tangency portfolio** — highest return per unit of risk.

$$\max_w \frac{w^T \mu - r_f}{\sqrt{w^T \Sigma w}}$$

subject to $\sum w_i = 1$ and $w_i \geq 0$ (long-only).

In [ ]:
# Max Sharpe Ratio optimization
ef_sharpe = EfficientFrontier(mu, cov_matrix, weight_bounds=(0, 0.25))  # Cap at 25% per asset
ef_sharpe.max_sharpe(risk_free_rate=risk_free_rate)

sharpe_weights = ef_sharpe.clean_weights(cutoff=0.01)
sharpe_perf = ef_sharpe.portfolio_performance(verbose=False, risk_free_rate=risk_free_rate)

print("=" * 60)
print("  MAX SHARPE RATIO PORTFOLIO")
print("=" * 60)
print(f"  Expected Return:  {sharpe_perf[0]:>8.2%}")
print(f"  Volatility:       {sharpe_perf[1]:>8.2%}")
print(f"  Sharpe Ratio:     {sharpe_perf[2]:>8.2f}")
print()
print(f"  {'Symbol':<8} {'Weight':>8}  {'Current':>8}  {'Change':>8}")
print(f"  {'-'*8} {'-'*8}  {'-'*8}  {'-'*8}")

for sym, w in sorted(sharpe_weights.items(), key=lambda x: -x[1]):
    if w > 0:
        cur_w = current_weights.get(sym, 0)
        delta = w - cur_w
        print(f"  {sym:<8} {w:>7.1%}   {cur_w:>7.1%}   {delta:>+7.1%}")

n_assets = sum(1 for w in sharpe_weights.values() if w > 0)
print(f"\n  Concentrated in {n_assets} assets (from {len(opt_symbols)} available)")
print("=" * 60)

### 9. Optimize: Minimum Volatility Portfolio

The **leftmost point** on the efficient frontier — lowest possible risk.

$$\min_w \sqrt{w^T \Sigma w}$$

In [ ]:
# Min Volatility optimization
ef_minvol = EfficientFrontier(mu, cov_matrix, weight_bounds=(0, 0.25))
ef_minvol.min_volatility()

minvol_weights = ef_minvol.clean_weights(cutoff=0.01)
minvol_perf = ef_minvol.portfolio_performance(verbose=False, risk_free_rate=risk_free_rate)

print("=" * 60)
print("  MINIMUM VOLATILITY PORTFOLIO")
print("=" * 60)
print(f"  Expected Return:  {minvol_perf[0]:>8.2%}")
print(f"  Volatility:       {minvol_perf[1]:>8.2%}")
print(f"  Sharpe Ratio:     {minvol_perf[2]:>8.2f}")
print()
print(f"  {'Symbol':<8} {'Weight':>8}")
print(f"  {'-'*8} {'-'*8}")

for sym, w in sorted(minvol_weights.items(), key=lambda x: -x[1]):
    if w > 0:
        print(f"  {sym:<8} {w:>7.1%}")

n_assets = sum(1 for w in minvol_weights.values() if w > 0)
print(f"\n  Diversified across {n_assets} assets")
print("=" * 60)

### 10. Efficient Frontier Visualization

Plot the risk-return tradeoff curve with the current portfolio,
max Sharpe, and min volatility portfolios marked.

In [ ]:
# Generate efficient frontier points using CLA
cla = CLA(mu, cov_matrix, weight_bounds=(0, 0.25))
(ef_returns, ef_vols, _) = cla.efficient_frontier(points=100)

# Create the plot
fig = go.Figure()

# Efficient frontier curve
fig.add_trace(go.Scatter(
    x=ef_vols, y=ef_returns,
    mode="lines",
    name="Efficient Frontier",
    line=dict(color="blue", width=2),
))

# Individual assets
asset_vols = np.sqrt(np.diag(cov_matrix))
fig.add_trace(go.Scatter(
    x=asset_vols, y=mu.values,
    mode="markers+text",
    name="Individual Assets",
    text=mu.index.tolist(),
    textposition="top center",
    textfont=dict(size=8),
    marker=dict(size=6, color="gray", opacity=0.6),
))

# Current portfolio
fig.add_trace(go.Scatter(
    x=[current_vol], y=[current_return],
    mode="markers+text",
    name=f"Current (Sharpe={current_sharpe:.2f})",
    text=["Current"],
    textposition="bottom right",
    marker=dict(size=14, color="red", symbol="star"),
))

# Max Sharpe portfolio
fig.add_trace(go.Scatter(
    x=[sharpe_perf[1]], y=[sharpe_perf[0]],
    mode="markers+text",
    name=f"Max Sharpe ({sharpe_perf[2]:.2f})",
    text=["Max Sharpe"],
    textposition="top left",
    marker=dict(size=14, color="green", symbol="diamond"),
))

# Min Volatility portfolio
fig.add_trace(go.Scatter(
    x=[minvol_perf[1]], y=[minvol_perf[0]],
    mode="markers+text",
    name=f"Min Vol (σ={minvol_perf[1]:.1%})",
    text=["Min Vol"],
    textposition="bottom left",
    marker=dict(size=14, color="orange", symbol="square"),
))

fig.update_layout(
    title="Efficient Frontier — My Portfolio",
    xaxis_title="Annual Volatility (Risk)",
    yaxis_title="Expected Annual Return",
    width=950, height=600,
    legend=dict(x=0.02, y=0.98),
    xaxis=dict(tickformat=".0%"),
    yaxis=dict(tickformat=".0%"),
)
fig.show()

### 11. Portfolio Comparison Summary

Side-by-side comparison of all three portfolios.

In [ ]:
comparison = pd.DataFrame({
    "Current": [current_return, current_vol, current_sharpe],
    "Max Sharpe": [sharpe_perf[0], sharpe_perf[1], sharpe_perf[2]],
    "Min Volatility": [minvol_perf[0], minvol_perf[1], minvol_perf[2]],
}, index=["Expected Return", "Volatility", "Sharpe Ratio"])

print("\n" + "=" * 60)
print("  PORTFOLIO COMPARISON")
print("=" * 60)
print(comparison.to_string(float_format=lambda x: f"{x:.4f}"))
print("=" * 60)

# Bar chart comparison
fig = go.Figure()
for col in comparison.columns:
    fig.add_trace(go.Bar(name=col, x=comparison.index, y=comparison[col].values))

fig.update_layout(
    barmode="group",
    title="Portfolio Comparison: Current vs Optimized",
    width=800, height=450,
    yaxis_title="Value",
)
fig.show()

### 12. Weight Comparison — Current vs Max Sharpe

Which positions should grow and which should shrink?

In [ ]:
# Build comparison DataFrame
weight_comp = pd.DataFrame({
    "Current": current_weights,
    "Max Sharpe": pd.Series(sharpe_weights),
}).fillna(0)

weight_comp["Delta"] = weight_comp["Max Sharpe"] - weight_comp["Current"]
weight_comp = weight_comp[weight_comp.abs().max(axis=1) > 0.005]  # Filter noise
weight_comp = weight_comp.sort_values("Delta")

# Diverging bar chart
fig = go.Figure()
colors = ["green" if d > 0 else "red" for d in weight_comp["Delta"]]
fig.add_trace(go.Bar(
    y=weight_comp.index,
    x=weight_comp["Delta"],
    orientation="h",
    marker_color=colors,
    text=[f"{d:+.1%}" for d in weight_comp["Delta"]],
    textposition="outside",
))

fig.update_layout(
    title="Rebalancing Direction: Current → Max Sharpe",
    xaxis_title="Weight Change",
    xaxis=dict(tickformat=".0%"),
    width=800, height=max(400, len(weight_comp) * 22),
    margin=dict(l=100),
)
fig.show()

### 13. Discrete Allocation (Whole Shares)

Given a total portfolio value, how many shares of each asset should we buy?
This converts fractional optimal weights into a real trade plan.

In [ ]:
# Total portfolio value for allocation
total_value = portfolio.loc[portfolio.index.isin(opt_symbols), "total_value"].sum()

# Latest prices for discrete allocation
latest_prices = get_latest_prices(prices)

# Discrete allocation for max Sharpe
da = DiscreteAllocation(sharpe_weights, latest_prices, total_portfolio_value=total_value)
allocation, leftover = da.greedy_portfolio()

print("=" * 60)
print(f"  DISCRETE ALLOCATION (Max Sharpe) — ${total_value:,.0f} portfolio")
print("=" * 60)
print(f"  {'Symbol':<8} {'Shares':>8} {'Price':>10} {'Value':>12} {'Weight':>8}")
print(f"  {'-'*8} {'-'*8} {'-'*10} {'-'*12} {'-'*8}")

alloc_total = 0
for sym, shares in sorted(allocation.items(), key=lambda x: -x[1] * latest_prices[x[0]]):
    price = latest_prices[sym]
    value = shares * price
    weight = value / total_value
    alloc_total += value
    print(f"  {sym:<8} {shares:>8} ${price:>9,.2f} ${value:>11,.2f} {weight:>7.1%}")

print(f"\n  Allocated: ${alloc_total:,.2f}")
print(f"  Leftover:  ${leftover:,.2f}")
print("=" * 60)

### 14. Per-Account Breakdown

Show the current allocation within each Fidelity account separately.
This is useful because:
- **Taxable accounts** — rebalancing triggers capital gains/losses
- **Retirement accounts** (401k, Roth IRA) — rebalancing is tax-free
- **529 / HSA** — limited investment options, separate optimization

In [ ]:
# Per-account summary
account_summary = (
    positions_df
    .groupby("account_name")
    .agg(
        symbols=("symbol", "nunique"),
        total_value=("total_value", "sum"),
        total_cost=("total_cost", "sum"),
        total_gl=("total_gl", "sum"),
    )
    .sort_values("total_value", ascending=False)
)

account_summary["gl_pct"] = np.where(
    account_summary["total_cost"] != 0,
    account_summary["total_gl"] / account_summary["total_cost"] * 100,
    0,
)
account_summary["portfolio_pct"] = account_summary["total_value"] / account_summary["total_value"].sum() * 100

print("=" * 90)
print("  PER-ACCOUNT SUMMARY")
print("=" * 90)
for acct, row in account_summary.iterrows():
    print(f"  {acct}")
    print(f"    Symbols: {row['symbols']:>3}   "
          f"Value: ${row['total_value']:>12,.2f}   "
          f"G/L: {row['gl_pct']:>+7.1f}%   "
          f"Portfolio: {row['portfolio_pct']:>5.1f}%")
print("=" * 90)

# Sunburst chart: Account → Symbol
sunburst_df = positions_df[["account_name", "symbol", "total_value"]].copy()
sunburst_df = sunburst_df[sunburst_df["total_value"] > 0]

fig = px.sunburst(
    sunburst_df,
    path=["account_name", "symbol"],
    values="total_value",
    title="Portfolio Structure: Account → Symbol",
    width=800, height=700,
)
fig.show()

### 15. Risk Decomposition — Correlation Heatmap

Understanding which assets move together helps explain portfolio risk.
High correlations reduce diversification benefits.

In [ ]:
# Daily returns
daily_returns = prices.pct_change().dropna()

# Correlation matrix (only top holdings for readability)
top_n = min(20, len(opt_symbols))
top_symbols = portfolio.index[:top_n].tolist()
top_in_prices = [s for s in top_symbols if s in daily_returns.columns]

corr = daily_returns[top_in_prices].corr()

fig = px.imshow(
    corr,
    text_auto=".2f",
    color_continuous_scale="RdBu_r",
    zmin=-1, zmax=1,
    title=f"Return Correlation Matrix (Top {len(top_in_prices)} Holdings)",
    width=800, height=700,
)
fig.show()

### 16. Rolling Performance — Cumulative Returns

Compare buy-and-hold of the current portfolio vs the optimized portfolio
over the historical period.

In [ ]:
# Cumulative returns for current vs optimal
current_w = current_weights.reindex(daily_returns.columns).fillna(0)
sharpe_w = pd.Series(sharpe_weights).reindex(daily_returns.columns).fillna(0)

current_port_returns = daily_returns @ current_w
optimal_port_returns = daily_returns @ sharpe_w

cumulative = pd.DataFrame({
    "Current Portfolio": (1 + current_port_returns).cumprod() - 1,
    "Max Sharpe (Optimal)": (1 + optimal_port_returns).cumprod() - 1,
})

fig = px.line(
    cumulative,
    title="Cumulative Returns: Current vs. Optimal Portfolio",
    labels={"value": "Cumulative Return", "variable": "Portfolio"},
    width=950, height=500,
)
fig.update_layout(
    yaxis=dict(tickformat=".0%"),
    legend=dict(x=0.02, y=0.98),
)
fig.show()

# Summary stats
print("\nBacktest Summary (full period):")
for col in cumulative.columns:
    total_ret = cumulative[col].iloc[-1]
    print(f"  {col}: {total_ret:+.1%} cumulative")

### 17. Key Takeaways & Next Steps

**Interpretation Guide:**
- If your current Sharpe ratio is **close to the max Sharpe** portfolio, your allocation is near-optimal
- If the efficient frontier shows your current position **well below the curve**, there is significant room for improvement
- The **discrete allocation** gives you a concrete trade plan (buy/sell whole shares)

**Caveats:**
- MPT assumes returns are normally distributed and past correlations persist — both are approximations
- Transaction costs, taxes, and wash-sale rules are NOT modeled
- 25% per-asset cap prevents extreme concentration but may limit returns

**Future Work:**
- Add **Black-Litterman** model for incorporating personal views
- Add **tax-aware rebalancing** (minimize taxable events in Individual account)
- Add **sector constraints** to ensure diversification across industries
- Incorporate **dividend yield** into expected returns
- Add **Monte Carlo simulation** for confidence intervals

In [ ]:
# Cleanup
conn.close()
print("Database connection closed.")